In [1]:
# ========================================
# CUSTOM CNN WITH SE ATTENTION FOR CHEST X-RAY PNEUMONIA/New updated
# ======================================== 
import os
import time
import copy
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

warnings.filterwarnings("ignore")

In [2]:
# =========================
# 1. Reproducibility + safe device setup
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

def get_safe_device():
    if torch.cuda.is_available():
        try:
            _ = torch.tensor([1.0]).to("cuda")
            torch.cuda.manual_seed_all(SEED)
            return torch.device("cuda")
        except Exception:
            pass
    return torch.device("cpu")

device = get_safe_device()
print("Using device:", device)


Using device: cuda


In [3]:
# =========================
# 2. Dataset path
# =========================
DATASET_ROOT = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray"
TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
VAL_DIR = os.path.join(DATASET_ROOT, "val")
TEST_DIR = os.path.join(DATASET_ROOT, "test")

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png")

# =========================
# 3. Load file paths
# =========================
def collect_image_paths_and_labels(root_dir):
    data = []
    classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
    class_to_idx = {cls_name: idx for idx, cls_name in enumerate(classes)}

    for cls_name in classes:
        cls_path = os.path.join(root_dir, cls_name)
        for file_name in os.listdir(cls_path):
            if file_name.lower().endswith(IMG_EXTENSIONS):
                file_path = os.path.join(cls_path, file_name)
                data.append([file_path, cls_name, class_to_idx[cls_name]])

    df = pd.DataFrame(data, columns=["file_path", "label_name", "label"])
    return df, class_to_idx

train_df, class_to_idx = collect_image_paths_and_labels(TRAIN_DIR)
val_df, _ = collect_image_paths_and_labels(VAL_DIR)
test_df, _ = collect_image_paths_and_labels(TEST_DIR)

idx_to_class = {v: k for k, v in class_to_idx.items()}
num_classes = len(class_to_idx)


In [4]:
# =========================
# 4. Reset indexes
# =========================
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# =========================
# 5. Show class distribution
# =========================
def show_distribution(df, name):
    counts = df["label_name"].value_counts()
    return counts

train_counts = show_distribution(train_df, "Train")
val_counts = show_distribution(val_df, "Validation")
test_counts = show_distribution(test_df, "Test")


In [5]:
# =========================
# 6. Aggressive Transforms
# =========================
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.90, 1.00)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(8),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.RandomAffine(5, shear=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.485, 0.485], std=[0.229, 0.229, 0.229])
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.485, 0.485], std=[0.229, 0.229, 0.229])
])

# =========================
# 7. Custom dataset
# =========================
class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image = Image.open(row["file_path"]).convert("L").convert("RGB")
        label = int(row["label"])

        if self.transform:
            image = self.transform(image)

        return image, label
